In [2]:
# --- 1. CONFIGURA TUS DATOS ---

# Reemplaza [TU_API_KEY] por tu clave real
API_KEY = "Tc2MjcyNDA5NTUzMA==gkLbDV"
# Reemplaza [TU_EMAIL] por el correo asociado a tu API Key
EMAIL = "jbaeza2017@udec.cl"

# Obtención listado de Estaciones Metereológicas

In [3]:
import requests
import json
import pandas as pd

# --- 2. CONSTRUYE LA PETICIÓN ---
SERVICIO = "getEstacionesRedEma"
URL_BASE = "https://climatologia.meteochile.gob.cl/application/servicios"
URL_CATALOGO = f"{URL_BASE}/{SERVICIO}?usuario={EMAIL}&token={API_KEY}"

print(f"Preparando la solicitud para obtener el catálogo de estaciones EMA...")

# --- 3. EJECUTA Y PROCESA LA RESPUESTA (FINALMENTE CORREGIDO) ---
try:
    response = requests.get(URL_CATALOGO)
    response.raise_for_status()
    data_json = response.json()

    # 🎯 CORRECCIÓN FINAL: Usamos la clave 'datosEstacion'
    lista_estaciones = data_json.get('datosEstacion', [])

    if not lista_estaciones:
        print("ERROR: La lista de estaciones está vacía. Revisa el JSON de diagnóstico anterior.")
        raise ValueError("La lista de estaciones no fue obtenida correctamente.")

    # Crea el DataFrame del Catálogo
    df_catalogo = pd.DataFrame(lista_estaciones)

    # Renombramos y filtramos columnas
    df_catalogo = df_catalogo.rename(columns={
        'codigoNacional': 'CodigoEstacion',
        'nombreEstacion': 'NombreEstacion',
        'region': 'RegionNumero',
        'NombreRegion': 'RegionNombre', # Notar que la clave es 'NombreRegion' (capital 'N') en el JSON
        'latitud': 'Latitud',
        'longitud': 'Longitud'
    })

    # Nos aseguramos de tener solo las columnas que vas a necesitar
    columnas_base = ['CodigoEstacion', 'NombreEstacion', 'RegionNumero', 'RegionNombre', 'Latitud', 'Longitud']
    df_catalogo = df_catalogo[[col for col in columnas_base if col in df_catalogo.columns]]

    print("\n--- ¡ÉXITO! Catálogo de estaciones obtenido. ---")
    print(f"Total de estaciones encontradas: {len(df_catalogo)}")
    print("\nPrimeras 5 estaciones del catálogo:")
    print(df_catalogo.head())

    # Guardamos el resultado global
    global ESTACIONES_A_PROCESAR
    ESTACIONES_A_PROCESAR = df_catalogo.copy()
    print("\n¡El catálogo ha sido guardado en la variable ESTACIONES_A_PROCESAR!")

except requests.exceptions.RequestException as e:
    print(f"Ocurrió un error en la petición: {e}")
except ValueError as e:
    print(f"ERROR AL PROCESAR: {e}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

Preparando la solicitud para obtener el catálogo de estaciones EMA...

--- ¡ÉXITO! Catálogo de estaciones obtenido. ---
Total de estaciones encontradas: 148

Primeras 5 estaciones del catálogo:
   CodigoEstacion         NombreEstacion  RegionNumero RegionNombre  \
0          170001       Visviri Tenencia            15                
1          180005  Chacalluta, Arica Ap.            15                
2          180017                  Putre            15                
3          180018   Defensa Civil, Arica            15                
4          180042  Cerro Sombrero, Arica            15                

     Latitud   Longitud  
0  -17.59500  -69.47750  
1  -18.35555  -70.34028  
2  -18.20000  -69.56250  
3  -18.49111  -70.30139  
4  -18.51250  -70.26556  

¡El catálogo ha sido guardado en la variable ESTACIONES_A_PROCESAR!


In [4]:
conteo_por_region = ESTACIONES_A_PROCESAR['RegionNumero'].value_counts(dropna=False)

print("\n--- Conteo de Estaciones por 'RegionNumero' (CORREGIDO) ---")
print("Este conteo confirma que la columna está llena y es usable:")
print("-" * 50)
print(conteo_por_region)
print("-" * 50)
print(f"Total de estaciones contadas: {conteo_por_region.sum()}")


--- Conteo de Estaciones por 'RegionNumero' (CORREGIDO) ---
Este conteo confirma que la columna está llena y es usable:
--------------------------------------------------
RegionNumero
12    27
13    19
5     18
10    11
4     10
9      9
2      9
11     7
8      6
14     6
3      5
15     5
6      5
7      4
16     4
1      3
Name: count, dtype: int64
--------------------------------------------------
Total de estaciones contadas: 148


# Prueba descarga data diaria de una estación en un año específico

In [5]:
import requests
import json
import pandas as pd # Útil para organizar la data en una tabla

# Estos son los parámetros específicos de la información que quieres:
CODIGO_ESTACION = "330020" # Ejemplo: Base Antártica Bernardo O'Higgins
ANIO = "2023"              # El año de los datos que deseas obtener

# --- 2. CONSTRUYE LA PETICIÓN ---

SERVICIO = "getTemperaturaHistoricaDiaria"
URL_BASE = "https://climatologia.meteochile.gob.cl/application/servicios"
URL_COMPLETA = f"{URL_BASE}/{SERVICIO}/{CODIGO_ESTACION}/{ANIO}?usuario={EMAIL}&token={API_KEY}"

print(f"Preparando la solicitud para el servicio: {SERVICIO} de la estación {CODIGO_ESTACION} en el año {ANIO}...")

# --- 3. EJECUTA LA PETICIÓN Y PROCESA LA RESPUESTA (CORREGIDO) ---

try:
    # Realiza la petición GET
    response = requests.get(URL_COMPLETA)

    # Verifica si hubo un error HTTP (4xx o 5xx)
    response.raise_for_status()

    # Convierte el texto de respuesta a un objeto de Python (Diccionario)
    data_json = response.json()

    # ****** CLAVE: Inspecciona la estructura devuelta ******
    print("\n--- JSON Completo Recibido (Diagnóstico) ---")
    print(json.dumps(data_json, indent=4))
    print("-------------------------------------------\n")

    # Intenta procesar la estructura de ÉXITO
    if data_json.get('hayDatos') is False:
        print("ADVERTENCIA: La API indica que 'hayDatos' es Falso.")
        if data_json.get('mensaje'):
            print(f"Mensaje de la API: {data_json['mensaje']}")
        # Si hay advertencia, salimos del procesamiento exitoso

    elif 'datosEstacion' in data_json:
        # Estructura de ÉXITO esperada
        print("--- ¡ÉXITO! Data recibida correctamente. ---")
        print(f"Estación: {data_json['datosEstacion']['nombreEstacion']} ({data_json['datosEstacion']['codigoNacional']})")
        print(f"Producto: {data_json['producto']}")
        print("-" * 40)

        # Opcional: Mostrar los datos del primer día para verificar
        # Esto puede fallar si 'datos' está vacío, por eso hay que ser cautelosos
        try:
            primer_dia = data_json['datos']['1']['1']
            print(f"Datos del 1 de enero de {ANIO}:")
            print(f"  Temperatura Máxima: {primer_dia.get('maxima', 'N/A')} °C")
            print(f"  Temperatura Mínima: {primer_dia.get('minima', 'N/A')} °C")
            print(f"  Temperatura Media: {primer_dia.get('media', 'N/A')} °C")
        except (KeyError, TypeError):
             print("Advertencia: No se pudieron extraer los datos diarios, aunque la estructura principal fue exitosa.")

    else:
        # Si no es False y tampoco tiene 'datosEstacion', es una estructura desconocida
        print("ERROR DE ESTRUCTURA: No se encontró 'hayDatos: False' ni 'datosEstacion'.")
        print("Revisa el JSON impreso arriba para entender la respuesta de la API.")


except requests.exceptions.HTTPError as errh:
    # Manejo de errores HTTP (4xx o 5xx)
    print(f"Error en la petición (Código {response.status_code}):")
    print("Verifica si el Código de Estación, el Año o tu API Key/Email son correctos.")
    print(f"Detalle del error: {errh}")

except Exception as e:
    # Otros errores (problemas de conexión, etc.)
    print(f"Ocurrió un error inesperado: {e}")
# --- 4. TRANSFORMACIÓN A DATAFRAME (Para análisis) ---

# Verifica que la estructura de datos existe
if data_json.get('hayDatos') and 'datos' in data_json:

    datos_temporales = []

    # Iterar sobre los meses (la clave '1', '2', '3', etc.)
    for mes, dias_en_mes in data_json['datos'].items():
        # Iterar sobre los días del mes (la clave '1', '2', '3', etc.)
        for dia, mediciones in dias_en_mes.items():

            # Crear un diccionario base con la información del día
            registro = {
                'Mes': int(mes),
                'Dia': int(dia),
                'Fecha_API': mediciones.get('momento'),
                'Temp_Media': mediciones.get('media'),
                'Temp_Maxima': mediciones.get('maxima'),
                'Temp_Minima': mediciones.get('minima'),
                'Datos_Contados': mediciones.get('numDatos')
            }
            datos_temporales.append(registro)

    # Crear el DataFrame de Pandas
    df = pd.DataFrame(datos_temporales)

    print("\n" + "="*50)
    print("DATOS CONVERTIDOS A DATAFRAME DE PANDAS:")
    print("="*50)

    # Mostrar las primeras 5 filas
    print(df.head())

    print("\n" + "-"*50)
    print("Información general del DataFrame:")
    df.info()

else:
    print("\nEl campo 'hayDatos' es falso o la estructura de 'datos' no fue encontrada para el procesamiento.")


Preparando la solicitud para el servicio: getTemperaturaHistoricaDiaria de la estación 330020 en el año 2023...

--- JSON Completo Recibido (Diagnóstico) ---
{
    "pais": "Chile",
    "organismo": "Direcci\u00f3n Meteorl\u00f3gica de Chile",
    "fechaCreacion": "10-11-2025 00:51",
    "producto": "Temperatura hist\u00f3rica diaria. Incluye temperatura media, m\u00ednima y m\u00e1xima ",
    "datosEstacion": {
        "codigoNacional": 330020,
        "codigoOMM": "85577",
        "codigoOACI": "SCQN",
        "nombreEstacion": "Quinta Normal, Santiago",
        "latitud": "-33.44500",
        "longitud": "-70.68278",
        "altura": 520,
        "region": 13,
        "nombreRegion": "Metropolitana de Santiago"
    },
    "hayDatos": true,
    "ano": 2023,
    "datos": {
        "1": {
            "1": {
                "momento": "01-01-2023",
                "media": 20.399999618530273,
                "maxima": 30.200000762939453,
                "fechaMax": "2023-01-01 17:08",
 

# Prueba Piloto de Descarga (Región 13: 2021-2025)

In [6]:
# Filtramos el DataFrame ESTACIONES_A_PROCESAR (ya corregido)
REGION_PILOTO = 13
df_piloto = ESTACIONES_A_PROCESAR[ESTACIONES_A_PROCESAR['RegionNumero'] == REGION_PILOTO].copy()

print(f"Catálogo de estaciones para la Región {REGION_PILOTO} (Piloto):")
print(f"Total de estaciones para la prueba: {len(df_piloto)}")
print(df_piloto[['CodigoEstacion', 'NombreEstacion']].head())

Catálogo de estaciones para la Región 13 (Piloto):
Total de estaciones para la prueba: 19
    CodigoEstacion                  NombreEstacion
47          330019  Eulogio SÃ¡nchez, Tobalaba Ad.
48          330020         Quinta Normal, Santiago
49          330021              Pudahuel Santiago 
52          330071                       Talagante
53          330075                   RÃ­o Clarillo


# Función "descargar_datos_historicos"

In [7]:
import pandas as pd
import requests
import json
import time

def descargar_datos_historicos(df_catalogo, inicio_ano, fin_ano, email, api_key):
    """
    Descarga datos de temperatura diaria (media, máxima, mínima)
    para todas las estaciones en el DataFrame de catálogo para el rango de años especificado.

    Args:
        df_catalogo (pd.DataFrame): DataFrame con el catálogo de estaciones (incluye CodigoEstacion y RegionNumero).
        inicio_ano (int): Año de inicio de la descarga (ej: 2021).
        fin_ano (int): Año de fin de la descarga (ej: 2025).
        email (str): Correo registrado en la API de MeteoChile.
        api_key (str): Token de API personal de MeteoChile.

    Returns:
        pd.DataFrame: DataFrame consolidado con todos los datos históricos descargados.
    """
    URL_BASE_TEMP = "https://climatologia.meteochile.gob.cl/application/servicios/getTemperaturaHistoricaDiaria"

    # Lista para almacenar los DataFrames de cada petición
    bd_historica = []

    # Definimos el rango de años
    anos = range(inicio_ano, fin_ano + 1)

    # CONTADOR GLOBAL para mostrar progreso
    total_peticiones = len(df_catalogo) * len(anos)
    peticiones_procesadas = 0

    # Bucle principal: Iterar sobre AÑOS
    for ano in anos:
        print(f"\n{'='*50}\n▶️ INICIANDO DESCARGA PARA EL AÑO {ano}\n{'='*50}")

        # Bucle secundario: Iterar sobre ESTACIONES
        for index, row in df_catalogo.iterrows():
            codigo_estacion = row['CodigoEstacion']

            # 1. CONSTRUIR URL DE PETICIÓN
            url_peticion = f"{URL_BASE_TEMP}/{codigo_estacion}/{ano}?usuario={email}&token={api_key}"

            peticiones_procesadas += 1

            # Formato de nombre amigable para el log
            nombre_estacion_log = row['NombreEstacion'].split(',')[0].strip()
            print(f"[{peticiones_procesadas}/{total_peticiones}] Descargando: {nombre_estacion_log} ({codigo_estacion}) - {ano}...")

            try:
                # 2. AQUI COMIENZA LA DESCARGA
                response = requests.get(url_peticion)
                response.raise_for_status() # Lanza error para códigos 4xx/5xx
                data_json = response.json()

                # 3. PROCESAMOS JSON DE TEMPERATURA
                # La estructura de la data de temperatura es anidada: {'datos': {'mes': {'dia': {temperaturas...}}}}
                if 'datos' in data_json and data_json['datos']:

                    datos_temporales = []
                    for mes, dias_en_mes in data_json['datos'].items():
                        for dia, mediciones in dias_en_mes.items():
                            registro = {
                                'Fecha_API': mediciones.get('momento'),
                                'Temp_Media': mediciones.get('media'),
                                'Temp_Maxima': mediciones.get('maxima'),
                                'Temp_Minima': mediciones.get('minima'),
                                'Num_Datos': mediciones.get('numDatos')
                            }
                            datos_temporales.append(registro)

                    df_temp = pd.DataFrame(datos_temporales)

                    # 4. AGREGAMOS METADATOS DE LA ESTACIÓN
                    # Agregamos las columnas de metadatos del catálogo
                    for col in df_catalogo.columns:
                        df_temp[col] = row[col]

                    bd_historica.append(df_temp)
                    print(f"   ✅ Éxito: {len(df_temp)} registros de temperatura agregados.")

                else:
                    print("   ❌ Advertencia: La API no reportó datos para esta estación/año.")

            except requests.exceptions.RequestException as e:
                print(f"   ❌ ERROR HTTP para {codigo_estacion}-{ano}: {e}")
            except Exception as e:
                print(f"   ❌ ERROR al procesar JSON para {codigo_estacion}-{ano}: {e}")

            # Retraso de 1 segundo para evitar el bloqueo
            time.sleep(1)

    # 5. CONCATENAMOS Y LIMPIAMOS EL DATAFRAME FINAL
    if bd_historica:
        df_final = pd.concat(bd_historica, ignore_index=True)

        # Limpieza de fechas
        df_final['Fecha'] = pd.to_datetime(df_final['Fecha_API'], format='%d-%m-%Y', errors='coerce')
        df_final = df_final.drop(columns=['Fecha_API']).dropna(subset=['Fecha'])

        # Limpieza de temperaturas (conversión a numérico)
        for col in ['Temp_Media', 'Temp_Maxima', 'Temp_Minima']:
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
            df_final[col] = df_final[col].round(2)

        print(f"\n\n{'*'*70}\nBASE DE DATOS HISTÓRICA CREADA CON ÉXITO.\n{'*'*70}")
        print(f"Registros totales: {len(df_final):,}")

        # Reordenar columnas para mejor visualización
        columnas_ordenadas = ['CodigoEstacion', 'NombreEstacion', 'RegionNumero', 'Fecha', 'Temp_Media', 'Temp_Maxima', 'Temp_Minima']
        otras_columnas = [col for col in df_final.columns if col not in columnas_ordenadas]
        df_final = df_final[columnas_ordenadas + otras_columnas]

        return df_final
    else:
        print("\nLa descarga terminó, pero no se recuperaron datos válidos.")
        return pd.DataFrame()

In [8]:
BD_PILOTO = descargar_datos_historicos(
    df_catalogo=df_piloto,
    inicio_ano=2021,
    fin_ano=2025,
    email=EMAIL,  # 🔑 REEMPLAZAR con tu correo
    api_key= API_KEY # 🔑 REEMPLAZAR con tu API Key
)

# Imprimir un resumen del resultado
if not BD_PILOTO.empty:
    print("\n--- ✅ Resumen de la Base de Datos Piloto ---")
    print(f"Total de registros de temperatura descargados: {len(BD_PILOTO):,}")
    print(f"Período de fechas: {BD_PILOTO['Fecha'].min().date()} a {BD_PILOTO['Fecha'].max().date()}")
    print("\nPrimeros registros (Datos de Temperatura y Región):")
    print(BD_PILOTO[['NombreEstacion', 'RegionNumero', 'Fecha', 'Temp_Media', 'Temp_Maxima']].head())

    # OPCIONAL: Calcular la temperatura media simple de la región
    temp_media_general = BD_PILOTO['Temp_Media'].mean()
    print(f"\nTemperatura media general para la Región 13 (2021-2025): {temp_media_general:.2f}°C")


▶️ INICIANDO DESCARGA PARA EL AÑO 2021
[1/95] Descargando: Eulogio SÃ¡nchez (330019) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[2/95] Descargando: Quinta Normal (330020) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[3/95] Descargando: Pudahuel Santiago (330021) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[4/95] Descargando: Talagante (330071) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[5/95] Descargando: RÃ­o Clarillo (330075) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[6/95] Descargando: Chorombo Hacienda (330076) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[7/95] Descargando: El Colorado (330077) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[8/95] Descargando: Polpaico (330081) - 2021...
   ❌ Advertencia: La API no reportó datos para esta estación/año.
[9/95] Descargando: Lo Prado Cerro San Francisco (330111) - 2021...
   ✅ Éxito: 365 registros de temperatu

Guardar archivo en excel o csv

In [11]:
# Definimos un nombre de archivo para el piloto
nombre_archivo_piloto = 'temperatura_piloto_region_13_2021_2025.xlsx'

# Verificamos si el DataFrame piloto existe y no está vacío
if 'BD_PILOTO' in locals() and not BD_PILOTO.empty:
    BD_PILOTO.to_excel(nombre_archivo_piloto, index=False)
    print(f"✅ ¡ÉXITO! Los datos piloto (Región 13) han sido guardados en el archivo: {nombre_archivo_piloto}")
    print(f"Total de registros guardados: {len(BD_PILOTO):,}")
else:
    print("❌ ERROR: La variable 'BD_PILOTO' no existe o está vacía. Asegúrate de ejecutar la prueba piloto primero.")

✅ ¡ÉXITO! Los datos piloto (Región 13) han sido guardados en el archivo: temperatura_piloto_region_13_2021_2025..xlsx
Total de registros guardados: 27,946


# DESCARGA DE DATOS MASIVOS BD HISTORICO T°


In [15]:
# --- EJECUCIÓN DEL PROCESO MASIVO CON GUARDADO AUTOMÁTICO EN EXCEL O CSV---
# Se utiliza el catálogo completo: ESTACIONES_A_PROCESAR (148 estaciones)
# Rango de años: 2021 a 2025 (740 peticiones)

print("=========================================================")
print("🚀 INICIANDO DESCARGA MASIVA DE DATOS (148 ESTACIONES)")
print("=========================================================")

# Ejecución de la función con el catálogo completo de estaciones
BD_HISTORICA_FINAL = descargar_datos_historicos(
    df_catalogo=ESTACIONES_A_PROCESAR,
    inicio_ano=2021,
    fin_ano=2025,
    email=EMAIL,  # 🔑 Se reemplaza correo cuenta Dirección Metereológica
    api_key= API_KEY # 🔑 Se reemplaza API Key entregada por la Dirección Metereológica
)


🚀 INICIANDO DESCARGA MASIVA DE DATOS (148 ESTACIONES)

▶️ INICIANDO DESCARGA PARA EL AÑO 2021
[1/740] Descargando: Visviri Tenencia (170001) - 2021...
   ❌ Advertencia: La API no reportó datos para esta estación/año.
[2/740] Descargando: Chacalluta (180005) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[3/740] Descargando: Putre (180017) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[4/740] Descargando: Defensa Civil (180018) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[5/740] Descargando: Cerro Sombrero (180042) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[6/740] Descargando: Colchane (190010) - 2021...
   ❌ Advertencia: La API no reportó datos para esta estación/año.
[7/740] Descargando: Diego Aracena Iquique Ap. (200006) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[8/740] Descargando: UNAP (Universidad Arturo Prat) (200010) - 2021...
   ✅ Éxito: 365 registros de temperatura agregados.
[9/740] Desca

/tmp/ipython-input-2912918939.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(bd_historica, ignore_index=True)




**********************************************************************
BASE DE DATOS HISTÓRICA CREADA CON ÉXITO.
**********************************************************************
Registros totales: 181,846


In [16]:
# Guardar la data automáticamente en formato CSV (.csv)
if not BD_HISTORICA_FINAL.empty:
    nombre_archivo_final_csv = 'temperatura_historica_chile_2021_2025.csv'

    # 💥 LÍNEA CORREGIDA: Sin paréntesis extra.
    BD_HISTORICA_FINAL.to_csv(nombre_archivo_final_csv, index=False, encoding='utf-8')

    print(f"\n🎉 ¡DESCARGA COMPLETA! Los datos han sido guardados en el archivo CSV: {nombre_archivo_final_csv}")
    print(f"Total de registros finales: {len(BD_HISTORICA_FINAL):,}")
    print("\nEstructura inicial del DataFrame final:")
    print(BD_HISTORICA_FINAL[['NombreEstacion', 'RegionNumero', 'Fecha', 'Temp_Media']].head())


🎉 ¡DESCARGA COMPLETA! Los datos han sido guardados en el archivo CSV: temperatura_historica_chile_2021_2025.csv
Total de registros finales: 181,846

Estructura inicial del DataFrame final:
          NombreEstacion  RegionNumero      Fecha  Temp_Media
0  Chacalluta, Arica Ap.            15 2021-01-01        22.3
1  Chacalluta, Arica Ap.            15 2021-01-02        22.8
2  Chacalluta, Arica Ap.            15 2021-01-03        23.3
3  Chacalluta, Arica Ap.            15 2021-01-04        22.6
4  Chacalluta, Arica Ap.            15 2021-01-05        22.3
